# Agent 2 — Step C: Train XGBoost + Evaluate + Inference Test

**Input:** `train.csv`, `test.csv`, `encoders.pkl`

**Output:** `price_model.pkl` (trained XGBoost) + evaluation metrics + a working `predict_price()` function.

**Why XGBoost (not LSTM):**
- Our data is monthly/quarterly, only ~1800 time points → LSTM overfits badly on small data
- XGBoost handles categorical + temporal features well and trains in seconds
- Target accuracy: MAPE < 15% (CLAUDE.md goal)
- We can add LSTM later as an ensemble if needed

## Cell 1 — Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd
import numpy as np
import pickle

DRIVE = "/content/drive/MyDrive/MilletSaarthi"

train = pd.read_csv(f"{DRIVE}/train.csv")
test  = pd.read_csv(f"{DRIVE}/test.csv")
with open(f"{DRIVE}/features.txt") as f:
    FEATURES = f.read().strip().split(",")
with open(f"{DRIVE}/encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

TARGET = "modal_price"
X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {FEATURES}")

## Cell 2 — Train XGBoost

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric="mae"
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
print("\n✅ Training done")

## Cell 3 — Evaluate on test set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
r2   = r2_score(y_test, preds)

print(f"MAE  : ₹{mae:.2f} per quintal")
print(f"RMSE : ₹{rmse:.2f} per quintal")
print(f"MAPE : {mape:.2f}%   (target: <15%)")
print(f"R²   : {r2:.4f}")

# Feature importance
print("\nFeature importance:")
imp = pd.DataFrame({"feature": FEATURES, "importance": model.feature_importances_})
print(imp.sort_values("importance", ascending=False).to_string(index=False))

## Cell 4 — Save model

In [ ]:
import os
os.makedirs(f"{DRIVE}/models", exist_ok=True)

with open(f"{DRIVE}/models/price_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ Saved models/price_model.pkl")
print("✅ encoders.pkl already saved from Step B")

## Cell 5 — Inference test (the function Agent 2 will use)

In [ ]:
def predict_price(millet: str, grade: str, state: str, district: str,
                  year: int, month: int, quantity_quintals: float = 1.0) -> dict:
    """
    Predict expected modal price per quintal.
    
    Args:
        millet: 'jowar' | 'bajra' | 'ragi'
        grade:  'A' | 'B' | 'C'  (from Agent 1)
        state:  e.g. 'Maharashtra'
        district: e.g. 'Pune'
        year, month: prediction month
        quantity_quintals: farmer's quantity (multiplied for total revenue)
    
    Returns: JSON dict for the Orchestrator.
    """
    def season_of(m):
        if m in (6, 7, 8, 9): return "kharif"
        if m in (10, 11, 12, 1, 2, 3): return "rabi"
        return "summer"

    row = {
        "millet_enc":   encoders["millet"].transform([millet])[0],
        "state_enc":    encoders["state"].transform([state])[0],
        "district_enc": encoders["district"].transform([district])[0],
        "grade_enc":    encoders["grade"].transform([grade])[0],
        "season_enc":   encoders["season"].transform([season_of(month)])[0],
        "year":         year,
        "month":        month,
        "month_sin":    np.sin(2 * np.pi * month / 12),
        "month_cos":    np.cos(2 * np.pi * month / 12),
    }
    X = pd.DataFrame([row])[FEATURES]
    price = float(model.predict(X)[0])

    return {
        "millet": millet,
        "grade": grade,
        "location": f"{district}, {state}",
        "month": f"{year}-{month:02d}",
        "expected_price_per_quintal": round(price, 2),
        "quantity_quintals": quantity_quintals,
        "expected_total_revenue": round(price * quantity_quintals, 2),
    }

# Test 1 — Grade A Jowar in Pune, May 2026, 10 quintals
print(predict_price("jowar", "A", "Maharashtra", "Pune", 2026, 5, 10))

# Test 2 — Grade C Jowar same place, same month
print(predict_price("jowar", "C", "Maharashtra", "Pune", 2026, 5, 10))

# Test 3 — Grade A Bajra in Ahmednagar
print(predict_price("bajra", "A", "Maharashtra", "Ahmednagar", 2026, 5, 5))

# Test 4 — Grade B Ragi in Bangalore
print(predict_price("ragi", "B", "Karnataka", "Bangalore", 2026, 6, 8))